In [1]:
# 16 digit binary number will be used to represent state where 0 is off and 1 is on and cell numbered as
# 0 1 2 3
# 4 5 6 7
# 8 9 10 11
# 12 13 14 15
# where number 0 corresponds to first digit and so on

lo_map = {
}
for s in range(0,2**16):
    t={}
    for a in range(0,16):
        if s==0:
            t[a]=[(1,s,0,True)]
        else:
            nx=s^(2**a+(0 if a%4==0 else 2**(a-1))+(0 if a%4==3 else 2**(a+1))+(0 if a//4==0 else 2**(a-4))+(0 if a//4==3 else 2**(a+4)))
            if nx==0:
                t[a]=[(1,nx,0,True)] #changing reward from 1,0 to 0,-1 to get lesser number of steps as more favourble
            else:
                t[a]=[(1,nx,-1,False)]
    lo_map[s]=t

In [2]:
import numpy as np

states = list(range(0,2**16))
terminal_state = 0
actions = list(range(16))
V = [-1e9] * (2**16)
V[terminal_state] = 0.0
theta = 1e-4
gamma = 0.9

In [3]:
while True:
    delta = 0
    V_old=V.copy()
    for s in states:
        if s == terminal_state:
            continue
        V[s] = float("-inf")
        for a in actions:
            p, nx, r, done = lo_map[s][a][0]
            val = r + gamma * V_old[nx] #since it is a deterministic mdp we dont have to sum
            V[s] = max(V[s], val)
        delta = max(delta, abs(V[s] - V_old[s]))
    if delta < theta: #since delta is integer we could also do delta==0
        break

In [4]:
policy = {}
for s in states:
    if s == terminal_state:
        policy[s] = None
        continue
    best = float("-inf")
    best_action = None
    for a in actions:
        transitions = lo_map[s][a]
        #since it is determinsitic we can easily write:
        p, nx, r, done = transitions[0]
        val = r + gamma * V[nx]
        if val > best:
            best = val
            best_action = a
    policy[s] = best_action